# E-Commerce Data Cleaning

## Project: E-Commerce Competitive Pricing & Product Intelligence

This notebook demonstrates the data cleaning and validation process
performed on the raw E-Commerce dataset collected through QuickCommerceAPI.

### Data Sources

The dataset contains product information collected from:

- Amazon
- Flipkart

### Cities Covered

- Bengaluru
- Delhi
- Mumbai
- Hyderabad
- Guwahati

### Initial Dataset

The raw master dataset contains 4,679 product observations.

### Objective

The objective of this notebook is to transform the raw API-collected
dataset into a reliable, analysis-ready dataset for:

- Exploratory Data Analysis
- Competitive Pricing Analysis
- Product Intelligence
- SQL Business Analysis
- Power BI Dashboard

### Cleaning Activities

The notebook covers:

1. Loading the raw dataset
2. Initial data inspection
3. Missing-value analysis
4. Data-type validation
5. Text standardization
6. Removing completely missing columns
7. Removing constant columns
8. Handling missing brand and variant values
9. Removing invalid price records
10. Recalculating discount percentage
11. Removing exact duplicates
12. Removing repeated search-query observations
13. Validating ratings
14. Identifying price outliers
15. Validating discount percentage
16. Final data-quality validation
17. Exporting the cleaned dataset

## Data Cleaning Business Rules

The cleaning decisions are based on the structure and quality of the
actual collected dataset.

### Missing Values

- `brand` → replace missing values with `Unknown`
- `variant` → replace missing values with `Not Specified`
- `rating` → preserve missing values
- `review_count` → preserve missing values

Missing rating does not mean a product has a zero-star rating.

### Completely Missing Columns

The following columns contain 100% missing values:

- `subcategory`
- `model`
- `seller`

These columns are removed because they contain no analytical information.

### Constant Columns

The following fields contain only one value:

- `availability`
- `inventory`

They are removed from the analytical dataset because they do not provide
variation for analysis.

### Invalid Prices

Records where:

- MRP <= 0
- Selling Price <= 0

are considered invalid for competitive pricing analysis and are removed.

### Duplicate Products

The same product can appear multiple times because it may be returned
for different search queries.

The business grain is therefore:

`platform + city + product_id`

Repeated observations at this grain are deduplicated.

### Price Outliers

Potential price outliers are identified using the IQR method but are NOT
automatically removed because high-priced products may be legitimate.

Instead, an outlier flag is created.

### Important

The original raw dataset is never overwritten.

In [1]:
# ================================================================
# IMPORT REQUIRED LIBRARIES
# ================================================================

import os
import numpy as np
import pandas as pd

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_rows",
    100
)

## 1. Load Raw Dataset

The raw E-Commerce master dataset is stored in:

`data/raw/ecommerce/ecommerce_master.csv`

The raw dataset is kept unchanged so that the complete data pipeline
can be reproduced later.

In [2]:
# ================================================================
# LOAD RAW E-COMMERCE DATA
# ================================================================

INPUT_FILE = "data/raw/ecommerce/ecommerce_master.csv"

df = pd.read_csv(
    INPUT_FILE
)

print("Raw dataset loaded successfully.")
print()
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

Raw dataset loaded successfully.

Rows    : 4679
Columns : 22


In [3]:
# ================================================================
# CREATE A WORKING COPY
# ================================================================

# Keep the original dataframe untouched.
# All cleaning operations will be performed on this copy.

df_clean = df.copy()

print(
    "Working copy created successfully."
)

Working copy created successfully.


## 2. Initial Data Inspection

Before cleaning, the dataset is inspected to understand:

- Column names
- Data types
- Missing values
- Duplicate records
- Numerical distributions
- Categorical distributions

In [4]:
# ================================================================
# DATASET STRUCTURE
# ================================================================

print("Dataset Shape:")
print(df_clean.shape)

print("\nColumn Names:")
print(df_clean.columns.tolist())

print("\nData Types:")
print(df_clean.dtypes)

Dataset Shape:
(4679, 22)

Column Names:
['platform', 'product_id', 'product_name', 'brand', 'category', 'subcategory', 'model', 'variant', 'mrp', 'selling_price', 'discount_pct', 'rating', 'review_count', 'availability', 'inventory', 'seller', 'product_url', 'search_query', 'city', 'latitude', 'longitude', 'scrape_timestamp']

Data Types:
platform             object
product_id           object
product_name         object
brand                object
category             object
subcategory         float64
model               float64
variant              object
mrp                 float64
selling_price       float64
discount_pct        float64
rating              float64
review_count        float64
availability           bool
inventory             int64
seller              float64
product_url          object
search_query         object
city                 object
latitude            float64
longitude           float64
scrape_timestamp     object
dtype: object


In [5]:
# ================================================================
# INITIAL MISSING VALUE ANALYSIS
# ================================================================

missing_summary = pd.DataFrame({

    "missing_count":
        df_clean.isna().sum(),

    "missing_percentage":
        (
            df_clean.isna()
            .mean()
            .mul(100)
            .round(2)
        )

})

missing_summary = (
    missing_summary
    .sort_values(
        "missing_count",
        ascending=False
    )
)

missing_summary

,missing_count,missing_percentage
subcategory,4679,100.00
model,4679,100.00
seller,4679,100.00
brand,1203,25.71
variant,805,17.20
discount_pct,53,1.13
rating,41,0.88
review_count,41,0.88
inventory,0,0.00
longitude,0,0.00


In [6]:
# ================================================================
# EXACT DUPLICATE CHECK
# ================================================================

exact_duplicates = (
    df_clean
    .duplicated()
    .sum()
)

print(
    f"Exact duplicate rows: {exact_duplicates}"
)

Exact duplicate rows: 0


In [7]:
# ================================================================
# STATISTICAL SUMMARY
# ================================================================

df_clean.describe(
    include="all"
).T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
platform,4679,2,Flipkart,3476,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,4679,3614,SMWGEH7VV8B3H8Y6,4,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_name,4679,3217,Noise Icon 4 with Stunning 1.96'' AMOLED Displ...,13,NaN,NaN,NaN,NaN,NaN,NaN,NaN
brand,3476,152,Samsung,350,NaN,NaN,NaN,NaN,NaN,NaN,NaN
category,4679,6,Smartphones,1060,NaN,NaN,NaN,NaN,NaN,NaN,NaN
subcategory,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
model,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
variant,3874,1559,8 GB RAM,281,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mrp,4679.0,NaN,NaN,NaN,51698.411985,156492.373684,0.0,9999.0,32999.0,69695.0,9999990.0
selling_price,4679.0,NaN,NaN,NaN,37269.092665,59933.894968,0.0,4499.0,23293.0,52990.0,2699990.0


## 3. Standardize Column Names

Column names are standardized into lowercase snake_case.

This makes the dataset easier to work with in:

- Python
- SQL
- Power BI
- Data pipelines

In [8]:
# ================================================================
# STANDARDIZE COLUMN NAMES
# ================================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)

print(
    df_clean.columns.tolist()
)

['platform', 'product_id', 'product_name', 'brand', 'category', 'subcategory', 'model', 'variant', 'mrp', 'selling_price', 'discount_pct', 'rating', 'review_count', 'availability', 'inventory', 'seller', 'product_url', 'search_query', 'city', 'latitude', 'longitude', 'scrape_timestamp']


## 4. Clean Text Columns

Text fields are cleaned by:

- Removing leading/trailing spaces
- Replacing empty strings with missing values
- Removing repeated whitespace
- Standardizing platform and city names

In [9]:
# ================================================================
# CLEAN TEXT COLUMNS
# ================================================================

text_columns = [
    "platform",
    "product_id",
    "product_name",
    "brand",
    "category",
    "subcategory",
    "model",
    "variant",
    "seller",
    "product_url",
    "search_query",
    "city"
]

for column in text_columns:

    if column not in df_clean.columns:
        continue

    # Convert empty/whitespace-only values to NaN.
    df_clean[column] = (
        df_clean[column]
        .replace(
            r"^\s*$",
            np.nan,
            regex=True
        )
    )

    # Remove leading and trailing spaces.
    if df_clean[column].dtype == "object":

        df_clean[column] = (
            df_clean[column]
            .astype("string")
            .str.strip()
        )

    # Replace repeated spaces.
    if pd.api.types.is_string_dtype(
        df_clean[column]
    ):

        df_clean[column] = (
            df_clean[column]
            .str.replace(
                r"\s+",
                " ",
                regex=True
            )
        )

# Standardize platform names.
df_clean["platform"] = (
    df_clean["platform"]
    .str.title()
)

# Standardize city names.
df_clean["city"] = (
    df_clean["city"]
    .str.title()
)

print("Text cleaning completed.")

Text cleaning completed.


## 5. Convert Data Types

Numeric fields are converted to numeric data types.

The timestamp field is converted to a datetime type.

This ensures that calculations and filtering work correctly.

In [10]:
# ================================================================
# CONVERT DATA TYPES
# ================================================================

numeric_columns = [
    "mrp",
    "selling_price",
    "discount_pct",
    "rating",
    "review_count",
    "inventory",
    "latitude",
    "longitude"
]

for column in numeric_columns:

    if column in df_clean.columns:

        df_clean[column] = pd.to_numeric(
            df_clean[column],
            errors="coerce"
        )

# Convert availability to boolean.
if "availability" in df_clean.columns:

    df_clean["availability"] = (
        df_clean["availability"]
        .astype("boolean")
    )

# Convert scrape timestamp to datetime.
if "scrape_timestamp" in df_clean.columns:

    df_clean["scrape_timestamp"] = (
        pd.to_datetime(
            df_clean["scrape_timestamp"],
            errors="coerce"
        )
    )

print(
    df_clean.dtypes
)

platform            string[python]
product_id          string[python]
product_name        string[python]
brand               string[python]
category            string[python]
subcategory                float64
model                      float64
variant             string[python]
mrp                        float64
selling_price              float64
discount_pct               float64
rating                     float64
review_count               float64
availability               boolean
inventory                    int64
seller                     float64
product_url         string[python]
search_query        string[python]
city                string[python]
latitude                   float64
longitude                  float64
scrape_timestamp    datetime64[ns]
dtype: object


## 6. Remove Completely Missing Columns

The initial data-quality analysis identified three columns with 100%
missing values:

- `subcategory`
- `model`
- `seller`

Since these columns contain no information, they are removed.

We do not fill them with `Unknown` because there are no observed values
to support meaningful analysis.

In [11]:
# ================================================================
# REMOVE 100% MISSING COLUMNS
# ================================================================

completely_missing_columns = [
    column
    for column in df_clean.columns
    if df_clean[column].isna().all()
]

print(
    "Completely missing columns:"
)

print(
    completely_missing_columns
)

df_clean = df_clean.drop(
    columns=completely_missing_columns
)

print("\nDataset shape after removal:")
print(df_clean.shape)

Completely missing columns:
['subcategory', 'model', 'seller']

Dataset shape after removal:
(4679, 19)


## 7. Remove Constant Columns

The initial analysis showed that:

- `availability` = True for all records
- `inventory` = 1 for all records

These fields have no variation and therefore provide no analytical
value for comparing products.

They are removed from the analysis dataset.

The original raw dataset remains unchanged.

In [12]:
# ================================================================
# REMOVE CONSTANT COLUMNS
# ================================================================

constant_columns = [
    column
    for column in df_clean.columns
    if df_clean[column].nunique(
        dropna=False
    ) <= 1
]

print(
    "Constant columns:"
)

print(
    constant_columns
)

df_clean = df_clean.drop(
    columns=constant_columns
)

print("\nDataset shape after removal:")
print(df_clean.shape)

Constant columns:
['availability', 'inventory']

Dataset shape after removal:
(4679, 17)


## 8. Handle Missing Brand Values

The raw dataset contains 1,203 missing brand values.

These rows are NOT removed because the other product information remains
useful for pricing and category analysis.

Instead:

`Missing brand → Unknown`

We do not infer brand from `search_query`, because the search query only
describes the query used to retrieve the product.

In [13]:
# ================================================================
# HANDLE MISSING BRAND VALUES
# ================================================================

missing_brand_before = (
    df_clean["brand"]
    .isna()
    .sum()
)

df_clean["brand"] = (
    df_clean["brand"]
    .fillna("Unknown")
)

print(
    f"Brand values replaced with 'Unknown': "
    f"{missing_brand_before}"
)

print("\nRemaining missing brands:")
print(
    df_clean["brand"].isna().sum()
)

Brand values replaced with 'Unknown': 1203

Remaining missing brands:
0


## 9. Handle Missing Variant Values

The raw dataset contains missing variant information.

These records are retained because the product itself can still be
analyzed.

Missing variant values are replaced with:

`Not Specified`

In [14]:
# ================================================================
# HANDLE MISSING VARIANT VALUES
# ================================================================

missing_variant_before = (
    df_clean["variant"]
    .isna()
    .sum()
)

df_clean["variant"] = (
    df_clean["variant"]
    .fillna("Not Specified")
)

print(
    f"Variant values replaced with 'Not Specified': "
    f"{missing_variant_before}"
)

print("\nRemaining missing variants:")
print(
    df_clean["variant"].isna().sum()
)

Variant values replaced with 'Not Specified': 805

Remaining missing variants:
0


## 10. Preserve Missing Ratings and Review Counts

There are a small number of products with missing:

- `rating`
- `review_count`

These values are not replaced with zero.

Reason:

`Missing ≠ Zero`

A missing rating means the source did not provide a rating. It does not
mean the product has a 0-star rating.

The missing values will therefore remain as NaN.

In [15]:
# ================================================================
# CHECK MISSING RATINGS AND REVIEW COUNTS
# ================================================================

print(
    "Missing ratings:",
    df_clean["rating"].isna().sum()
)

print(
    "Missing review counts:",
    df_clean["review_count"].isna().sum()
)

Missing ratings: 41
Missing review counts: 41


## 11. Validate Prices

For competitive pricing analysis, both MRP and selling price must be
greater than zero.

The raw dataset contains 53 observations where pricing information is
invalid.

Business rule:

- MRP > 0
- Selling Price > 0

Invalid price records will be removed.

In [16]:
# ================================================================
# IDENTIFY INVALID PRICE RECORDS
# ================================================================

invalid_price_mask = (
    (df_clean["mrp"] <= 0)
    |
    (df_clean["selling_price"] <= 0)
)

invalid_price_records = (
    df_clean[invalid_price_mask]
    .copy()
)

print(
    "Invalid price records:",
    len(invalid_price_records)
)

invalid_price_records[
    [
        "platform",
        "product_id",
        "product_name",
        "category",
        "mrp",
        "selling_price",
        "city",
        "search_query"
    ]
].head(20)

Invalid price records: 53


,platform,product_id,product_name,category,mrp,selling_price,city,search_query
23,Flipkart,MOBH9AS47XHFRMJY,"Samsung Galaxy F06 5G (Bahama Blue, 128 GB)",Smartphones,0.0,0.0,Bengaluru,Samsung smartphone
247,Amazon,B0GYWQB116,"17T (Black,12GB+256GB)|Flagship Leica Cameras|...",Smartphones,0.0,0.0,Bengaluru,Xiaomi smartphone
264,Flipkart,COMHDZVEH5XGAYBF,HP 15 (i5 14th Gen) Intel Core 5 120U - (16 GB...,Laptops,0.0,0.0,Bengaluru,HP laptop
485,Amazon,B0G92LZJD3,"Vivobook 15 (2025), 13th Gen,Intel Core i3-131...",Laptops,0.0,0.0,Bengaluru,ASUS laptop
912,Amazon,B0F43CHDSN,138 cm (55 inches) Vision AI 4K Ultra HD Smart...,Televisions,0.0,0.0,Bengaluru,Samsung TV
1446,Flipkart,COMH2TPS6EYHZMHF,"Acer Aspire Lite with Backlit Keyboard, Intel ...",Laptops,0.0,0.0,Delhi,Acer laptop
1707,Flipkart,ACCHHKT3YM47HUBD,GDS Collapsible Bluetooth Bass Pulse Signature...,Headphones/Earbuds,0.0,0.0,Delhi,Bose headphones
1708,Flipkart,ACCHHHYYXJTGG2KK,GDS Foldable Wireless Bass Boost Pro Audio_FW ...,Headphones/Earbuds,0.0,0.0,Delhi,Bose headphones
1709,Flipkart,ACCHGWFCGZ4ZWJYZ,"GDS Bluetooth Headphones, Foldable Wireless De...",Headphones/Earbuds,0.0,0.0,Delhi,Bose headphones
1710,Flipkart,ACCHGWFBXUHX9HDF,GDS Bass-Boost Wireless Headphones with Mic & ...,Headphones/Earbuds,0.0,0.0,Delhi,Bose headphones


In [17]:
# ================================================================
# REMOVE INVALID PRICE RECORDS
# ================================================================

rows_before_price_cleaning = (
    len(df_clean)
)

df_clean = (
    df_clean[
        ~invalid_price_mask
    ]
    .copy()
)

rows_after_price_cleaning = (
    len(df_clean)
)

print(
    "Rows before price cleaning:",
    rows_before_price_cleaning
)

print(
    "Rows after price cleaning:",
    rows_after_price_cleaning
)

print(
    "Rows removed:",
    rows_before_price_cleaning
    -
    rows_after_price_cleaning
)

Rows before price cleaning: 4679
Rows after price cleaning: 4626
Rows removed: 53


## 12. Validate Selling Price Against MRP

Selling price should not exceed MRP in the collected pricing data.

The validation checks for:

`selling_price > mrp`

No records are automatically removed based on this check because the
purpose is first to identify potential source-data issues.

In [18]:
# ================================================================
# SELLING PRICE VS MRP VALIDATION
# ================================================================

selling_price_above_mrp = (
    df_clean["selling_price"]
    >
    df_clean["mrp"]
)

print(
    "Selling price > MRP:",
    selling_price_above_mrp.sum()
)

Selling price > MRP: 0


## 13. Recalculate Discount Percentage

The discount percentage is recalculated from the cleaned MRP and
selling price values.

Formula:

Discount % =
((MRP - Selling Price) / MRP) × 100

This ensures that the discount metric is internally consistent with the
cleaned price fields.

In [19]:
# ================================================================
# RECALCULATE DISCOUNT PERCENTAGE
# ================================================================

df_clean["discount_pct"] = (

    (
        df_clean["mrp"]
        -
        df_clean["selling_price"]
    )
    /
    df_clean["mrp"]

) * 100

df_clean["discount_pct"] = (
    df_clean["discount_pct"]
    .round(2)
)

print(
    df_clean["discount_pct"].describe()
)

count    4626.000000
mean       34.770914
std        25.546680
min         0.000000
25%        12.955000
50%        31.920000
75%        51.440000
max        97.220000
Name: discount_pct, dtype: float64


In [20]:
# ================================================================
# VALIDATE DISCOUNT RANGE
# ================================================================

invalid_discount_mask = (
    (df_clean["discount_pct"] < 0)
    |
    (df_clean["discount_pct"] > 100)
)

print(
    "Invalid discount records:",
    invalid_discount_mask.sum()
)

Invalid discount records: 0


## 14. Remove Exact Duplicate Rows

Exact duplicates are records where every column has exactly the same
value.

The initial dataset contained zero exact duplicate rows, but the
duplicate-removal step is still performed to make the pipeline robust
and reusable.

In [21]:
# ================================================================
# REMOVE EXACT DUPLICATES
# ================================================================

rows_before_duplicates = (
    len(df_clean)
)

df_clean = (
    df_clean
    .drop_duplicates()
    .reset_index(drop=True)
)

exact_duplicates_removed = (
    rows_before_duplicates
    -
    len(df_clean)
)

print(
    "Exact duplicates removed:",
    exact_duplicates_removed
)

Exact duplicates removed: 0


## 15. Remove Repeated Product Observations

The API may return the same product for multiple search queries.

For example, the same product can appear when searching for:

- Samsung smartphone
- OnePlus smartphone
- Xiaomi smartphone

The product itself remains the same, but `search_query` changes.

Therefore, the analytical business grain is:

`platform + city + product_id`

This prevents the same marketplace product from being counted multiple
times within the same city.

We retain one observation for each unique combination.

In [22]:
# ================================================================
# CHECK PRODUCT-LEVEL DUPLICATES
# ================================================================

business_key = [
    "platform",
    "city",
    "product_id"
]

product_duplicate_mask = (
    df_clean
    .duplicated(
        subset=business_key,
        keep=False
    )
)

product_duplicates = (
    df_clean[
        product_duplicate_mask
    ]
    .sort_values(
        business_key
    )
)

print(
    "Rows involved in product-level duplicates:",
    len(product_duplicates)
)

product_duplicates[
    [
        "platform",
        "city",
        "product_id",
        "product_name",
        "mrp",
        "selling_price",
        "discount_pct",
        "search_query"
    ]
].head(30)

Rows involved in product-level duplicates: 96


,platform,city,product_id,product_name,mrp,selling_price,discount_pct,search_query
532,Amazon,Bengaluru,B0BS1RT9S2,WH-CH520 Wireless Bluetooth Headphones On Ear ...,5990.0,3955.0,33.97,Sony headphones
667,Amazon,Bengaluru,B0BS1RT9S2,WH-CH520 Wireless Bluetooth Headphones On Ear ...,5990.0,3955.0,33.97,JBL headphones
897,Amazon,Bengaluru,B0F84FBWQM,80 cm (32 inches) HD Smart LED TV UA32H4550FUXXL,17900.0,15490.0,13.46,Samsung TV
964,Amazon,Bengaluru,B0F84FBWQM,80 cm (32 inches) HD Smart LED TV UA32H4550FUXXL,17900.0,15490.0,13.46,LG TV
51,Amazon,Bengaluru,B0G81P4MPG,"Galaxy M17 5G Mobile (Sapphire Black, 6GB RAM,...",23999.0,18999.0,20.83,Samsung smartphone
250,Amazon,Bengaluru,B0G81P4MPG,"Galaxy M17 5G Mobile (Sapphire Black, 6GB RAM,...",23999.0,18999.0,20.83,Xiaomi smartphone
50,Amazon,Bengaluru,B0G81TPT89,"Galaxy M17 5G Mobile (Moonlight Silver, 6GB RA...",23999.0,18999.0,20.83,Samsung smartphone
183,Amazon,Bengaluru,B0G81TPT89,"Galaxy M17 5G Mobile (Moonlight Silver, 6GB RA...",23999.0,18999.0,20.83,OnePlus smartphone
1084,Amazon,Bengaluru,B0G8J8SKTH,"189 L, 5 Star, Digital Inverter, Direct-Cool S...",23999.0,18290.0,23.79,LG refrigerator
1134,Amazon,Bengaluru,B0G8J8SKTH,"189 L, 5 Star, Digital Inverter, Direct-Cool S...",23999.0,18290.0,23.79,Samsung refrigerator


### Why Are These Duplicates?

These records are not necessarily erroneous duplicates.

They are mainly caused by the same product being returned for multiple
search queries.

For business analysis, the product should be counted once within each:

`platform + city`

combination.

Therefore, repeated search-query observations are removed.

In [23]:
# ================================================================
# REMOVE REPEATED SEARCH-QUERY OBSERVATIONS
# ================================================================

rows_before_product_deduplication = (
    len(df_clean)
)

df_clean = (
    df_clean
    .drop_duplicates(
        subset=business_key,
        keep="first"
    )
    .reset_index(drop=True)
)

product_duplicates_removed = (
    rows_before_product_deduplication
    -
    len(df_clean)
)

print(
    "Repeated product/search observations removed:",
    product_duplicates_removed
)

print(
    "Rows after product-level deduplication:",
    len(df_clean)
)

Repeated product/search observations removed: 49
Rows after product-level deduplication: 4577


## 16. Validate Product Ratings

Ratings are expected to be within the range:

`0 to 5`

Missing ratings are allowed because the source may not provide a rating.

In [24]:
# ================================================================
# RATING VALIDATION
# ================================================================

invalid_rating_mask = (
    df_clean["rating"].notna()
    &
    (
        (df_clean["rating"] < 0)
        |
        (df_clean["rating"] > 5)
    )
)

print(
    "Invalid rating records:",
    invalid_rating_mask.sum()
)

print("\nRating summary:")

print(
    df_clean["rating"].describe()
)

Invalid rating records: 0

Rating summary:
count    4536.000000
mean        3.825772
std         1.203266
min         0.000000
25%         4.000000
50%         4.200000
75%         4.400000
max         5.000000
Name: rating, dtype: float64


## 17. Identify Price Outliers

Potential price outliers are identified using the Interquartile Range
(IQR) method.

### Formula

IQR = Q3 - Q1

Lower Bound = Q1 - 1.5 × IQR

Upper Bound = Q3 + 1.5 × IQR

### Important Business Decision

Potential outliers are NOT removed automatically.

For example, a ₹2 lakh premium laptop may be a legitimate product
rather than an error.

Therefore, the dataset retains these observations and creates an
`selling_price_outlier` flag.

In [25]:
# ================================================================
# PRICE OUTLIER DETECTION USING IQR
# ================================================================

price = (
    df_clean["selling_price"]
    .dropna()
)

Q1 = price.quantile(
    0.25
)

Q3 = price.quantile(
    0.75
)

IQR = (
    Q3 - Q1
)

lower_bound = (
    Q1 - 1.5 * IQR
)

upper_bound = (
    Q3 + 1.5 * IQR
)

df_clean["selling_price_outlier"] = (

    (df_clean["selling_price"] < lower_bound)
    |
    (df_clean["selling_price"] > upper_bound)

)

print(
    f"Q1: ₹{Q1:,.2f}"
)

print(
    f"Q3: ₹{Q3:,.2f}"
)

print(
    f"IQR: ₹{IQR:,.2f}"
)

print(
    f"Lower Bound: ₹{lower_bound:,.2f}"
)

print(
    f"Upper Bound: ₹{upper_bound:,.2f}"
)

print(
    "Potential price outliers:",
    df_clean["selling_price_outlier"].sum()
)

Q1: ₹4,999.00
Q3: ₹53,490.00
IQR: ₹48,491.00
Lower Bound: ₹-67,737.50
Upper Bound: ₹126,226.50
Potential price outliers: 183


In [26]:
# ================================================================
# INSPECT POTENTIAL PRICE OUTLIERS
# ================================================================

outliers = (
    df_clean[
        df_clean["selling_price_outlier"]
    ]
    [
        [
            "platform",
            "city",
            "category",
            "brand",
            "product_name",
            "mrp",
            "selling_price"
        ]
    ]
    .sort_values(
        "selling_price",
        ascending=False
    )
)

outliers.head(30)

,platform,city,category,brand,product_name,mrp,selling_price
2427,Flipkart,Mumbai,Televisions,TCL,TCL 291 cm (115 inch) Ultra HD (4K) Mini LED S...,9999990.0,2699990.0
2424,Flipkart,Mumbai,Televisions,TCL,TCL C755 248 cm (98 inch) Ultra HD (4K) Mini L...,899990.0,899990.0
1615,Amazon,Delhi,Laptops,Unknown,2026 MacBook Pro Laptop with M5 Max chip with ...,619900.0,589990.0
1619,Amazon,Delhi,Laptops,Unknown,2026 MacBook Pro Laptop with M5 Max chip with ...,539900.0,513990.0
1575,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Max, 2026) M5 Max - (36 ...",499900.0,499900.0
1585,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Pro, 2026) M5 Pro - (24 ...",399900.0,399900.0
1487,Flipkart,Delhi,Laptops,MSI,MSI Raider 18 HX Intel Core i9 14th Gen 14900H...,379990.0,379990.0
1603,Flipkart,Delhi,Laptops,Apple,"Apple MacBook Pro (M5 Pro, 2026) M5 Pro - (24 ...",359900.0,359900.0
312,Flipkart,Bengaluru,Laptops,ASUS,ASUS ExpertBook Ultra Copilot+ PC Intel Core U...,389990.0,352990.0
1591,Flipkart,Delhi,Laptops,Apple,Apple M4 Max - (48 GB/256 GB SSD/macOS Sequoia...,399900.0,348490.0


## 18. Platform Distribution

The collected dataset contains observations from:

- Amazon
- Flipkart

The distribution is not perfectly balanced because the API returned
different numbers of products for different searches.

We do not artificially balance the dataset by deleting or duplicating
records.

In [27]:
# ================================================================
# PLATFORM DISTRIBUTION
# ================================================================

platform_distribution = (
    df_clean["platform"]
    .value_counts()
)

platform_distribution

platform
Flipkart    3416
Amazon      1161
Name: count, dtype: Int64

In [28]:
# ================================================================
# CITY DISTRIBUTION
# ================================================================

city_distribution = (
    df_clean["city"]
    .value_counts()
)

city_distribution

city
Bengaluru    1155
Delhi        1063
Guwahati     1036
Hyderabad    1032
Mumbai        291
Name: count, dtype: Int64

In [29]:
# ================================================================
# PLATFORM × CITY DISTRIBUTION
# ================================================================

platform_city = pd.crosstab(
    df_clean["city"],
    df_clean["platform"]
)

platform_city

platform,Amazon,Flipkart
city,,
Bengaluru,307,848
Delhi,266,797
Guwahati,261,775
Hyderabad,267,765
Mumbai,60,231


## 19. Category Distribution

The dataset contains products across multiple categories.

The category distribution is checked after cleaning to ensure that the
cleaning process did not unexpectedly remove an entire product category.

In [30]:
# ================================================================
# CATEGORY DISTRIBUTION
# ================================================================

category_distribution = (
    df_clean["category"]
    .value_counts()
)

category_distribution

category
Smartphones           1041
Laptops                895
Headphones/Earbuds     877
Smartwatches           774
Televisions            615
Home Appliances        375
Name: count, dtype: Int64

## 20. Final Missing-Value Analysis

After cleaning, the remaining missing values are reviewed.

Expected behavior:

- `brand` → no missing values
- `variant` → no missing values
- `rating` → some missing values may remain
- `review_count` → some missing values may remain

Rating and review-count missing values are intentionally preserved.

In [31]:
# ================================================================
# FINAL MISSING VALUE ANALYSIS
# ================================================================

final_missing = pd.DataFrame({

    "missing_count":
        df_clean.isna().sum(),

    "missing_percentage":
        (
            df_clean.isna()
            .mean()
            .mul(100)
            .round(2)
        )

})

final_missing = (
    final_missing
    .sort_values(
        "missing_count",
        ascending=False
    )
)

final_missing

,missing_count,missing_percentage
rating,41,0.9
review_count,41,0.9
product_id,0,0.0
scrape_timestamp,0,0.0
longitude,0,0.0
latitude,0,0.0
city,0,0.0
search_query,0,0.0
product_url,0,0.0
platform,0,0.0


## 21. Final Data-Type Validation

In [32]:
# ================================================================
# FINAL DATA TYPES
# ================================================================

df_clean.dtypes

platform                 string[python]
product_id               string[python]
product_name             string[python]
brand                    string[python]
category                 string[python]
variant                  string[python]
mrp                             float64
selling_price                   float64
discount_pct                    float64
rating                          float64
review_count                    float64
product_url              string[python]
search_query             string[python]
city                     string[python]
latitude                        float64
longitude                       float64
scrape_timestamp         datetime64[ns]
selling_price_outlier              bool
dtype: object

## 22. Final Duplicate Validation

After product-level deduplication, the business grain should be unique:

`platform + city + product_id`

No duplicate combinations should remain.

In [33]:
# ================================================================
# FINAL DUPLICATE VALIDATION
# ================================================================

remaining_product_duplicates = (
    df_clean
    .duplicated(
        subset=[
            "platform",
            "city",
            "product_id"
        ]
    )
    .sum()
)

print(
    "Remaining product-level duplicates:",
    remaining_product_duplicates
)

Remaining product-level duplicates: 0


## 23. Final Dataset Overview

In [34]:
# ================================================================
# FINAL DATASET OVERVIEW
# ================================================================

print("=" * 70)
print("FINAL CLEANED DATASET")
print("=" * 70)

print(
    f"Rows    : {df_clean.shape[0]}"
)

print(
    f"Columns : {df_clean.shape[1]}"
)

print("\nColumns:")

print(
    df_clean.columns.tolist()
)

print("\nFirst 5 records:")

display(
    df_clean.head()
)

FINAL CLEANED DATASET
Rows    : 4577
Columns : 18

Columns:
['platform', 'product_id', 'product_name', 'brand', 'category', 'variant', 'mrp', 'selling_price', 'discount_pct', 'rating', 'review_count', 'product_url', 'search_query', 'city', 'latitude', 'longitude', 'scrape_timestamp', 'selling_price_outlier']

First 5 records:


,platform,product_id,product_name,brand,category,variant,mrp,selling_price,discount_pct,rating,review_count,product_url,search_query,city,latitude,longitude,scrape_timestamp,selling_price_outlier
0,Flipkart,MOBHMGHQ7MHMMHJZ,"Samsung Galaxy F70e 5G (Limelight Green, 128 GB)",Samsung,Smartphones,4 GB RAM,18999.0,14999.0,21.05,4.2,1049.0,https://www.flipkart.com/-/p/-?pid=MOBHMGHQ7MH...,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648735,False
1,Flipkart,MOBHCTXH9NNZEUVH,"Samsung Galaxy M06 5G (Sage Green, 64 GB)",Samsung,Smartphones,4 GB RAM,12499.0,9999.0,20.00,4.2,108.0,https://www.flipkart.com/-/p/-?pid=MOBHCTXH9NN...,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648765,False
2,Flipkart,MOBHMGHHXHYZEAVB,"Samsung Galaxy F07 (Green, 64 GB)",Samsung,Smartphones,4 GB RAM,11999.0,11499.0,4.17,4.3,655.0,https://www.flipkart.com/-/p/-?pid=MOBHMGHHXHY...,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648776,False
3,Flipkart,MOBHZ8YSUXXHYGBR,"Samsung Galaxy F70 Pro 5G (Alpha Black, 128 GB)",Samsung,Smartphones,8 GB RAM,49999.0,29999.0,40.00,4.3,36.0,https://www.flipkart.com/-/p/-?pid=MOBHZ8YSUXX...,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648783,False
4,Flipkart,MOBHMGHQF8KKB99Q,"Samsung Galaxy F70e 5G (Spotlight Blue, 128 GB)",Samsung,Smartphones,6 GB RAM,20999.0,16999.0,19.05,4.2,760.0,https://www.flipkart.com/-/p/-?pid=MOBHMGHQF8K...,Samsung smartphone,Bengaluru,12.9716,77.5946,2026-08-14 17:39:58.648789,False


## 24. Final Data Quality Checks

The final dataset should satisfy the following conditions:

- No completely missing columns
- No constant columns
- No invalid prices
- No selling price greater than MRP
- No invalid ratings
- No invalid discount percentages
- No exact duplicates
- No duplicate `platform + city + product_id`
- Missing ratings are preserved
- Potential price outliers are flagged rather than deleted

In [36]:
# ================================================================
# FINAL DATA QUALITY CHECKS
# ================================================================

print("=" * 70)
print("FINAL DATA QUALITY VALIDATION")
print("=" * 70)

# ---------------------------------------------------------------
# Check invalid prices.
# ---------------------------------------------------------------

invalid_prices = (
    (
        df_clean["mrp"] <= 0
    )
    |
    (
        df_clean["selling_price"] <= 0
    )
).sum()

print(
    f"Invalid price records       : {invalid_prices}"
)

# ---------------------------------------------------------------
# Check selling price > MRP.
# ---------------------------------------------------------------

price_above_mrp = (
    df_clean["selling_price"]
    >
    df_clean["mrp"]
).sum()

print(
    f"Selling price > MRP         : {price_above_mrp}"
)

# ---------------------------------------------------------------
# Check ratings.
# ---------------------------------------------------------------

invalid_ratings = (
    (
        df_clean["rating"] < 0
    )
    |
    (
        df_clean["rating"] > 5
    )
).sum()

print(
    f"Invalid ratings             : {invalid_ratings}"
)

# ---------------------------------------------------------------
# Check discounts.
# ---------------------------------------------------------------

invalid_discounts = (
    (
        df_clean["discount_pct"] < 0
    )
    |
    (
        df_clean["discount_pct"] > 100
    )
).sum()

print(
    f"Invalid discounts           : {invalid_discounts}"
)

# ---------------------------------------------------------------
# Check exact duplicates.
# ---------------------------------------------------------------

exact_duplicates = (
    df_clean
    .duplicated()
    .sum()
)

print(
    f"Exact duplicates            : {exact_duplicates}"
)

# ---------------------------------------------------------------
# Check business duplicates.
# ---------------------------------------------------------------

business_duplicates = (
    df_clean
    .duplicated(
        subset=[
            "platform",
            "city",
            "product_id"
        ]
    )
    .sum()
)

print(
    f"Business duplicates         : {business_duplicates}"
)

FINAL DATA QUALITY VALIDATION
Invalid price records       : 0
Selling price > MRP         : 0
Invalid ratings             : 0
Invalid discounts           : 0
Exact duplicates            : 0
Business duplicates         : 0


## 25. Save Cleaned Dataset

The cleaned dataset is saved to:

`data/processed/ecommerce_cleaned.csv`

The raw dataset is not overwritten.

In [38]:
# ================================================================
# SAVE CLEANED DATASET
# ================================================================

OUTPUT_DIR = "data/processed"

OUTPUT_FILE = os.path.join(
    OUTPUT_DIR,
    "ecommerce_cleaned.csv"
)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

df_clean.to_csv(
    OUTPUT_FILE,
    index=False
)

print(
    "Cleaned dataset saved successfully."
)

print(
    f"File: {OUTPUT_FILE}"
)

print(
    f"Rows: {len(df_clean)}"
)

Cleaned dataset saved successfully.
File: data/processed\ecommerce_cleaned.csv
Rows: 4577


## 26. Cleaning Summary

The raw E-Commerce dataset has now been transformed into an
analysis-ready dataset.

### Major transformations

| Area | Action |
|---|---|
| Completely missing columns | Removed |
| Constant columns | Removed |
| Missing brand | Replaced with `Unknown` |
| Missing variant | Replaced with `Not Specified` |
| Missing rating | Preserved |
| Missing review count | Preserved |
| Invalid prices | Removed |
| Discount | Recalculated |
| Exact duplicates | Removed |
| Search-query duplicates | Removed |
| Invalid ratings | Validated |
| Price outliers | Flagged, not removed |
| Platform/city imbalance | Preserved and documented |

### Next Step

The cleaned dataset will be used for:

1. Exploratory Data Analysis
2. Business KPI calculation
3. Competitive pricing analysis
4. Product and category analysis
5. City-level analysis
6. Platform comparison
7. Business recommendations
8. Power BI dashboard